<a href="https://colab.research.google.com/github/srishti-06069/ITA-/blob/main/Water%20leakage%3A%20Random%20Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
=============================================================================
  Water Pipeline Vibration Sensor — Leak Detection & Classification
  Based on: Lee & Kim (2023), Sensors 23(21):8935
  DOI: https://doi.org/10.3390/s23218935
  Dataset: AI Hub Korea — https://aihub.or.kr (dataSetSn=138)

  NOTE: This script generates synthetic data that EXACTLY mirrors the
  class distribution, sample counts, feature structure, and pipe types
  from the paper's Table 1. You can replace the synthetic data section
  with your real CSVs once downloaded from AI Hub.

  Real data link (free registration required):
  https://www.aihub.or.kr/aihubdata/data/view.do?
  currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=138
=============================================================================
"""

# ── 0. Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, IsolationForest, ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, accuracy_score, f1_score)
from sklearn.decomposition import PCA

try:
    import xgboost as xgb;   XGB_OK = True
except ImportError:
    XGB_OK = False;  print("[INFO] xgboost not found — pip install xgboost")

try:
    import lightgbm as lgb;  LGB_OK = True
except ImportError:
    LGB_OK = False;  print("[INFO] lightgbm not found — pip install lightgbm")

OUT = "aihub_output"
os.makedirs(OUT, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# COLOUR PALETTE
# ─────────────────────────────────────────────────────────────────────────────
PALETTE = {
    'out':    '#E53935',   # Outdoor leak   — red
    'in':     '#FB8C00',   # Indoor leak    — orange
    'noise':  '#FDD835',   # Electric noise — yellow
    'other':  '#8E24AA',   # Other noise    — purple
    'normal': '#1565C0',   # Normal         — blue
}
CLASS_LABELS = {
    'out':    'Outdoor Leak',
    'in':     'Indoor Leak',
    'noise':  'Electric Noise',
    'other':  'Other Noise',
    'normal': 'Normal (No Leak)',
}
CLASS_ORDER = ['out', 'in', 'noise', 'other', 'normal']

# ─────────────────────────────────────────────────────────────────────────────
# 1.  SYNTHETIC DATA GENERATION  (mirrors Table 1 exactly)
# ─────────────────────────────────────────────────────────────────────────────
# The real sensor produces vibration readings across ~32 frequency bands
# (10 Hz → 5120 Hz, in octave sub-bands).
# We simulate 32 frequency-band energy features per sample.

np.random.seed(42)

# Exact counts from the paper's Table 1
COUNTS = {
    'train': {'out': 17539, 'in': 13273, 'noise': 5029, 'other': 7019, 'normal': 19704},
    'test':  {'out':  2192, 'in':  1659, 'noise':  629, 'other':  878, 'normal':  2462},
}

N_FREQ_BANDS = 32          # frequency sub-bands (Hz)
FREQ_LABELS  = [f"f_{int(10 * 2**(i/2))}hz" for i in range(N_FREQ_BANDS)]

# Each class has a distinct frequency-energy signature:
#   outdoor leak  → high energy at low & mid bands (50–500 Hz)
#   indoor leak   → moderate energy, broadband
#   electric noise→ sharp spike at high bands (1000–5000 Hz)
#   other noise   → irregular across all bands
#   normal        → low, uniform energy
CLASS_PROFILES = {
    #            (mean_vector_seed, signal_band_start, signal_band_end, amplitude, noise_std)
    'out':    dict(bands=(4, 14),  amp=3.5, base=0.5, noise=0.6),
    'in':     dict(bands=(8, 20),  amp=2.5, base=0.4, noise=0.7),
    'noise':  dict(bands=(22, 31), amp=4.0, base=0.3, noise=0.5),
    'other':  dict(bands=(2, 28),  amp=1.5, base=0.5, noise=1.0),
    'normal': dict(bands=(0,  0),  amp=0.0, base=0.3, noise=0.3),
}

# Also simulate pipe_type as a categorical feature
PIPE_TYPES = ['SP', 'STS', 'DCIP', 'PE', 'PVC']

def generate_class_samples(cls, n):
    p   = CLASS_PROFILES[cls]
    X   = np.random.normal(p['base'], p['noise'], size=(n, N_FREQ_BANDS))
    # add signature energy bump in the characteristic band range
    s, e = p['bands']
    if e > s:
        X[:, s:e] += np.random.normal(p['amp'], p['noise']*0.5,
                                       size=(n, e - s))
    X = np.clip(X, 0, None)                  # energy cannot be negative
    # add pipe type as two numeric features (one-hot would bloat; use ordinal)
    pipe_idx = np.random.choice(len(PIPE_TYPES), size=n)
    metallic  = (pipe_idx < 3).astype(float) + np.random.normal(0, 0.05, n)
    pipe_feat = pipe_idx / 4.0               # normalised 0–1
    return np.column_stack([X, metallic, pipe_feat])

def build_split(split_name):
    frames, labels = [], []
    for cls in CLASS_ORDER:
        n = COUNTS[split_name][cls]
        X = generate_class_samples(cls, n)
        frames.append(X)
        labels.extend([cls] * n)
    X_all = np.vstack(frames)
    feature_names = FREQ_LABELS + ['is_metallic', 'pipe_type_norm']
    df = pd.DataFrame(X_all, columns=feature_names)
    df['label'] = labels
    # shuffle
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

print("=" * 70)
print("  AI Hub Korea — Water Pipeline Vibration Leak Detection")
print("  Paper: Lee & Kim (2023), Sensors 23(21):8935")
print("  DOI  : https://doi.org/10.3390/s23218935")
print("=" * 70)
print("\n[1] Generating synthetic dataset (mirrors Table 1 class counts) ...")

train_df = build_split('train')
test_df  = build_split('test')

print(f"    Train shape : {train_df.shape}  (total {len(train_df):,})")
print(f"    Test  shape : {test_df.shape}  (total {len(test_df):,})")
print(f"\n    Train class distribution:")
for cls in CLASS_ORDER:
    n = (train_df['label'] == cls).sum()
    print(f"      {CLASS_LABELS[cls]:<22} : {n:>6,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2] Preprocessing ...")

FEATURE_COLS = [c for c in train_df.columns if c != 'label']
X_train_raw  = train_df[FEATURE_COLS].values
X_test_raw   = test_df[FEATURE_COLS].values
y_train_str  = train_df['label'].values
y_test_str   = test_df['label'].values

le = LabelEncoder()
le.fit(CLASS_ORDER)
y_train = le.transform(y_train_str)
y_test  = le.transform(y_test_str)

# Binary: 1 = any leak class, 0 = normal
y_train_bin = (y_train_str != 'normal').astype(int)
y_test_bin  = (y_test_str  != 'normal').astype(int)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f"    Features : {len(FEATURE_COLS)} ({N_FREQ_BANDS} freq bands + 2 pipe features)")
print(f"    Classes  : {list(le.classes_)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3] Training models ...")
results = {}

# ── Random Forest ─────────────────────────────────────────────────────────────
print("    → Random Forest ...")
rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
yp = rf.predict(X_test)
results['Random Forest'] = dict(acc=accuracy_score(y_test, yp),
    f1=f1_score(y_test, yp, average='weighted'),
    cm=confusion_matrix(y_test, yp), y_pred=yp, model=rf, binary=False)
print(f"       Accuracy : {results['Random Forest']['acc']:.4f}")

# ── Extra Trees ───────────────────────────────────────────────────────────────
print("    → Extra Trees ...")
et = ExtraTreesClassifier(n_estimators=200, n_jobs=-1, random_state=42)
et.fit(X_train, y_train)
yp = et.predict(X_test)
results['Extra Trees'] = dict(acc=accuracy_score(y_test, yp),
    f1=f1_score(y_test, yp, average='weighted'),
    cm=confusion_matrix(y_test, yp), y_pred=yp, model=et, binary=False)
print(f"       Accuracy : {results['Extra Trees']['acc']:.4f}")

# ── XGBoost ───────────────────────────────────────────────────────────────────
if XGB_OK:
    print("    → XGBoost ...")
    xgb_m = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                eval_metric='mlogloss', n_jobs=-1, random_state=42,
                                verbosity=0)
    xgb_m.fit(X_train, y_train)
    yp = xgb_m.predict(X_test)
    results['XGBoost'] = dict(acc=accuracy_score(y_test, yp),
        f1=f1_score(y_test, yp, average='weighted'),
        cm=confusion_matrix(y_test, yp), y_pred=yp, model=xgb_m, binary=False)
    print(f"       Accuracy : {results['XGBoost']['acc']:.4f}")

# ── LightGBM ──────────────────────────────────────────────────────────────────
if LGB_OK:
    print("    → LightGBM ...")
    lgb_m = lgb.LGBMClassifier(n_estimators=300, num_leaves=63,
                                 learning_rate=0.1, n_jobs=-1,
                                 random_state=42, verbose=-1)
    lgb_m.fit(X_train, y_train)
    yp = lgb_m.predict(X_test)
    results['LightGBM'] = dict(acc=accuracy_score(y_test, yp),
        f1=f1_score(y_test, yp, average='weighted'),
        cm=confusion_matrix(y_test, yp), y_pred=yp, model=lgb_m, binary=False)
    print(f"       Accuracy : {results['LightGBM']['acc']:.4f}")

# ── Isolation Forest (unsupervised — your original concept) ──────────────────
print("    → Isolation Forest (unsupervised, binary) ...")
contamination = float(min(round(y_train_bin.mean(), 3), 0.499))
iso = IsolationForest(n_estimators=200, contamination=contamination,
                      random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_scores  = iso.decision_function(X_test)
iso_pred    = (iso.predict(X_test) == -1).astype(int)
results['Isolation Forest'] = dict(
    acc=accuracy_score(y_test_bin, iso_pred),
    f1=f1_score(y_test_bin, iso_pred, average='weighted'),
    cm=confusion_matrix(y_test_bin, iso_pred),
    y_pred=iso_pred, model=iso, binary=True, scores=iso_scores)
print(f"       Accuracy (binary) : {results['Isolation Forest']['acc']:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  CLASSIFICATION REPORTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  CLASSIFICATION REPORTS")
print("=" * 70)
for name, res in results.items():
    print(f"\n── {name} ──")
    if res['binary']:
        print(classification_report(y_test_bin, res['y_pred'],
                                     target_names=['Normal', 'Leak']))
    else:
        print(classification_report(y_test, res['y_pred'],
                                     target_names=le.classes_))

# ─────────────────────────────────────────────────────────────────────────────
# 5.  VISUALISATIONS
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5] Generating figures ...")
GREY = '#F0F4F8';  DARK = '#0D2B4E'

# ── Fig A: Class distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='white')
fig.suptitle("AI Hub Korea — Water Pipe Vibration Dataset\nClass Distribution (Lee & Kim, 2023 · Sensors 23(21):8935)",
             fontsize=12, fontweight='bold', color=DARK)

for ax, (split, df_s) in zip(axes, [('Train (62,564)', train_df), ('Test (7,820)', test_df)]):
    ax.set_facecolor(GREY)
    counts = [( df_s['label'] == c).sum() for c in CLASS_ORDER]
    bars   = ax.bar([CLASS_LABELS[c] for c in CLASS_ORDER], counts,
                    color=[PALETTE[c] for c in CLASS_ORDER],
                    edgecolor=DARK, linewidth=0.6)
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60,
                f"{cnt:,}", ha='center', va='bottom', fontsize=8.5,
                fontweight='bold', color=DARK)
    ax.set_title(split, fontsize=11, color=DARK)
    ax.set_ylabel("Number of Samples", fontsize=10)
    ax.tick_params(axis='x', rotation=20, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')

fig.tight_layout()
fig.savefig(f"{OUT}/A_class_distribution.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/A_class_distribution.png")

# ── Fig B: Frequency band energy profiles by class ────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5), facecolor='white')
ax.set_facecolor(GREY)
for cls in CLASS_ORDER:
    subset = train_df[train_df['label'] == cls][FREQ_LABELS].values
    mean   = subset.mean(axis=0)
    std    = subset.std(axis=0)
    x      = np.arange(N_FREQ_BANDS)
    ax.plot(x, mean, color=PALETTE[cls], lw=2.0,
            label=CLASS_LABELS[cls], zorder=3)
    ax.fill_between(x, mean - std, mean + std,
                    color=PALETTE[cls], alpha=0.12)

ax.set_xticks(range(0, N_FREQ_BANDS, 2))
ax.set_xticklabels([FREQ_LABELS[i] for i in range(0, N_FREQ_BANDS, 2)],
                   rotation=45, ha='right', fontsize=7.5)
ax.set_xlabel("Frequency Band", fontsize=11)
ax.set_ylabel("Mean Vibration Energy (normalised)", fontsize=11)
ax.set_title("Frequency-Band Energy Profiles per Leak Class\n"
             "Water Pipeline Vibration Sensor — AI Hub Korea Dataset",
             fontsize=12, fontweight='bold', color=DARK)
ax.legend(fontsize=9, framealpha=0.9)
for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')
fig.tight_layout()
fig.savefig(f"{OUT}/B_frequency_profiles.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/B_frequency_profiles.png")

# ── Fig C: Model accuracy & F1 comparison ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')
fig.suptitle("Supervised Model Performance Comparison\nAI Hub Water Pipeline Leak Detection",
             fontsize=12, fontweight='bold', color=DARK)

model_names = list(results.keys())
accs = [results[m]['acc'] for m in model_names]
f1s  = [results[m]['f1']  for m in model_names]
bar_colors = ['#1565C0','#00897B','#E53935','#FB8C00','#8E24AA'][:len(model_names)]

for ax, vals, metric in zip(axes, [accs, f1s], ['Accuracy', 'Weighted F1-Score']):
    ax.set_facecolor(GREY)
    bars = ax.bar(model_names, vals, color=bar_colors,
                  edgecolor=DARK, linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{v:.4f}", ha='center', va='bottom', fontsize=9,
                fontweight='bold', color=DARK)
    ax.set_ylim(0, 1.1)
    ax.set_title(metric, fontsize=11, color=DARK)
    ax.set_ylabel(metric, fontsize=10)
    ax.tick_params(axis='x', rotation=18, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')

fig.tight_layout()
fig.savefig(f"{OUT}/C_model_comparison.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/C_model_comparison.png")

# ── Fig D: Confusion matrices (supervised models only) ────────────────────────
sup = {k: v for k, v in results.items() if not v['binary']}
n = len(sup)
fig, axes = plt.subplots(1, n, figsize=(6*n, 5), facecolor='white')
if n == 1: axes = [axes]
fig.suptitle("Confusion Matrices — 5-Class Leak Classification",
             fontsize=12, fontweight='bold', color=DARK)
for ax, (name, res) in zip(axes, sup.items()):
    disp = ConfusionMatrixDisplay(res['cm'], display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=35)
    ax.set_title(f"{name}\nAcc={res['acc']:.4f} | F1={res['f1']:.4f}",
                 fontsize=10, color=DARK)
fig.tight_layout()
fig.savefig(f"{OUT}/D_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/D_confusion_matrices.png")

# ── Fig E: Feature importances (best model) ───────────────────────────────────
best = max(sup, key=lambda k: sup[k]['acc'])
imp  = results[best]['model'].feature_importances_
top_n = 20
idx  = np.argsort(imp)[::-1][:top_n]
cols_top  = [FEATURE_COLS[i] for i in idx]
imp_top   = imp[idx]
grad      = plt.cm.Blues(np.linspace(0.35, 0.9, top_n))[::-1]

fig, ax = plt.subplots(figsize=(10, 6), facecolor='white')
ax.set_facecolor(GREY)
ax.barh(cols_top[::-1], imp_top[::-1], color=grad,
        edgecolor=DARK, linewidth=0.4)
ax.set_xlabel("Feature Importance Score", fontsize=11)
ax.set_title(f"Top {top_n} Feature Importances — {best}\n"
             "Frequency-Band Vibration Features (AI Hub Korea Dataset)",
             fontsize=12, fontweight='bold', color=DARK)
for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')
fig.tight_layout()
fig.savefig(f"{OUT}/E_feature_importance.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/E_feature_importance.png")

# ── Fig F: Isolation Forest score distribution ───────────────────────────────
iso_res = results['Isolation Forest']
fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor(GREY)
ax.hist(iso_res['scores'][y_test_bin==0], bins=60, color='#1565C0',
        alpha=0.65, density=True, label='Normal (no leak)', edgecolor='none')
ax.hist(iso_res['scores'][y_test_bin==1], bins=60, color='#E53935',
        alpha=0.75, density=True, label='Leak (all types)',  edgecolor='none')
ax.axvline(iso.offset_, color=DARK, lw=1.8, ls='--',
           label=f'Decision threshold ({iso.offset_:.3f})')
ax.set_xlabel("Isolation Forest Anomaly Score", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Isolation Forest Anomaly Score — Binary Leak vs Normal\n"
             "Water Pipeline Vibration Sensor · AI Hub Korea",
             fontsize=12, fontweight='bold', color=DARK)
ax.legend(fontsize=10)
for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')
fig.tight_layout()
fig.savefig(f"{OUT}/F_isolation_forest.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/F_isolation_forest.png")

# ── Fig G: PCA 2D scatter — all 5 classes ────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
idx_s = np.random.choice(len(X_test), size=min(4000, len(X_test)), replace=False)
X_proj = pca.fit_transform(X_test[idx_s])
y_proj = y_test_str[idx_s]

fig, ax = plt.subplots(figsize=(9, 7), facecolor='white')
ax.set_facecolor(GREY)
for cls in CLASS_ORDER:
    m = y_proj == cls
    if m.sum() == 0: continue
    ax.scatter(X_proj[m, 0], X_proj[m, 1],
               c=PALETTE[cls], label=CLASS_LABELS[cls],
               s=20, alpha=0.55, edgecolors='none')
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
ax.set_title("PCA — 5-Class Separation of Frequency-Band Features\n"
             "Water Pipeline Vibration Sensor · AI Hub Korea",
             fontsize=12, fontweight='bold', color=DARK)
ax.legend(fontsize=10, framealpha=0.9)
ax.text(0.02, 0.02, f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%",
        transform=ax.transAxes, fontsize=9, color='#555',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')
fig.tight_layout()
fig.savefig(f"{OUT}/G_pca_scatter.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/G_pca_scatter.png")

# ── Fig H: Per-class F1 grouped bar chart ────────────────────────────────────
from sklearn.metrics import f1_score as f1_per

fig, ax = plt.subplots(figsize=(12, 5), facecolor='white')
ax.set_facecolor(GREY)
x   = np.arange(len(CLASS_ORDER))
w   = 0.18
sup_names = list(sup.keys())

for i, name in enumerate(sup_names):
    per_f1 = f1_score(y_test, results[name]['y_pred'], average=None,
                      labels=le.transform(CLASS_ORDER))
    offset = (i - len(sup_names)/2 + 0.5) * w
    ax.bar(x + offset, per_f1, w,
           label=name, edgecolor=DARK, linewidth=0.5,
           color=bar_colors[i])

ax.set_xticks(x)
ax.set_xticklabels([CLASS_LABELS[c] for c in CLASS_ORDER], fontsize=9.5)
ax.set_ylabel("F1-Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title("Per-Class F1-Score by Model\nAI Hub Korea — Water Pipeline Leak Classification",
             fontsize=12, fontweight='bold', color=DARK)
ax.legend(fontsize=9, framealpha=0.9)
ax.axhline(1.0, color=DARK, lw=0.8, ls='--', alpha=0.4)
for sp in ax.spines.values(): sp.set_edgecolor('#CBD5E0')
fig.tight_layout()
fig.savefig(f"{OUT}/H_per_class_f1.png", dpi=150, bbox_inches='tight')
plt.close();  print(f"  Saved: {OUT}/H_per_class_f1.png")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  RESULTS SUMMARY")
print("=" * 70)
print(f"  {'Model':<22} {'Accuracy':>10} {'Wtd F1':>10}  {'Task'}")
print(f"  {'-'*65}")
for name, res in results.items():
    task = 'Binary (Leak/Normal)' if res['binary'] else '5-Class'
    print(f"  {name:<22} {res['acc']:>10.4f} {res['f1']:>10.4f}  {task}")
print(f"\n  Benchmark (Lee & Kim, 2023): XGBoost → 99.79% on real dataset")
print(f"  Output saved to: ./{OUT}/   (8 figures)")
print("=" * 70)

  AI Hub Korea — Water Pipeline Vibration Leak Detection
  Paper: Lee & Kim (2023), Sensors 23(21):8935
  DOI  : https://doi.org/10.3390/s23218935

[1] Generating synthetic dataset (mirrors Table 1 class counts) ...
    Train shape : (62564, 35)  (total 62,564)
    Test  shape : (7820, 35)  (total 7,820)

    Train class distribution:
      Outdoor Leak           : 17,539
      Indoor Leak            : 13,273
      Electric Noise         :  5,029
      Other Noise            :  7,019
      Normal (No Leak)       : 19,704

[2] Preprocessing ...
    Features : 34 (32 freq bands + 2 pipe features)
    Classes  : [np.str_('in'), np.str_('noise'), np.str_('normal'), np.str_('other'), np.str_('out')]

[3] Training models ...
    → Random Forest ...
       Accuracy : 0.9999
    → Extra Trees ...
       Accuracy : 1.0000
    → XGBoost ...
       Accuracy : 1.0000
    → LightGBM ...
       Accuracy : 0.9999
    → Isolation Forest (unsupervised, binary) ...
       Accuracy (binary) : 0.8155

  C